Zadanie 1 - Wstępne przetwarzanie obrazów do modelu klasyfikacji (CIFAR-10 / ImageNet)

In [16]:
import torch

# Symulacja 16 obrazów BGR z OpenCV
bgr_image = torch.randint(0, 256, (16, 224, 224, 3), dtype=torch.float32)

# Wektory normujące dla kanałów R, G, B
mean = torch.tensor([0.485, 0.456, 0.406])
std = torch.tensor([0.229, 0.224, 0.225])

# Skalowanie wartości do przedziału [0.0, 0.1]
images_scaled = (bgr_image / 255.0)

# Odwrócenie kolejności kanałów z BRG na RGB
rgb_images = images_scaled[..., [2, 1, 0]]

# Permutacja osi z NHWC na NCHW
nchw_images = rgb_images.permute(0, 3, 1, 2)

# Standaryzacja z użyciem wektorów mean oraz std za pomocą broadcastingu
mean_view = mean.view(1, 3, 1, 1)
std_view = std.view(1, 3, 1, 1)

images_normalized = (nchw_images - mean_view) / std_view

print(f"Oczekiwany kształt wyjściowy: {images_normalized.shape}")

=== Podsumowanie ===
Oczekiwany kształt wyjściowy: torch.Size([16, 3, 224, 224])


Zadanie 2 - Obliczanie pola przecięcia ramek

In [18]:
# Dane z zadania
B, N, M = 2, 100, 10

P = torch.rand(B, N, 4)
G = torch.rand(B, M, 4)

# Rozszerzenie z użyciem broadcastingu
P_expanded = P.unsqueeze(2) # z [B, N, 4] robi [B, N, 1, 4]
G_expanded = G.unsqueeze(1) # z [B, M, 4] robi [B, 1, M, 4]

# Znajdujemy rogi nakładania się,
inter_x1 = torch.max(P_expanded[..., 0], G_expanded[..., 0])
inter_y1 = torch.max(P_expanded[..., 1], G_expanded[..., 1])
inter_x2 = torch.min(P_expanded[..., 2], G_expanded[..., 2])
inter_y2 = torch.min(P_expanded[..., 3], G_expanded[..., 3])

# Obliczanie szerokości i wysokości przecięcia
width = torch.clamp(inter_x2 - inter_x1, min=0)
height = torch.clamp(inter_y2 - inter_x1, min=0)

# Obliczanie pola przecięcia
intersection = width * height

print(f"Kształt wyniku: {intersection.shape}")

Kształt wyniku: torch.Size([2, 100, 10])


Zadanie 3 - Wyliczenie Macierzy kowariancji cech za pomocą einsum

In [20]:
# Dane z zadania
B, C, N = 4, 64, 1000

# Wygenerowanie tensora
X = torch.randn(B, C, N)

# Średnia po ostatnim wymiarze
mean = X.mean(dim=-1, keepdim=True)

# Wycentrowana macierz
X_centered = X - mean

# Użycie notacji einsum
# b c i -> to pierwszy tensor (wycentrowany)
# b d i -> to drugi tensor (ten sam wycentrowany)
# b d c -> wynikowy tensor gdzie wymiar 'i' (N) jest zredukowany
covariance_unscaled = torch.einsum('b c i, b d i -> b c d', X_centered, X_centered)

# Przeskalowywanie wyniku
covariance_matrix = covariance_unscaled / (N - 1)

print("=== PODSUMOWANIE WYMIARÓW ===")
print(f"Kształt tensora wejściowego X:        {X.shape}")
print(f"Kształt tensora średniej:             {mean.shape}")
print(f"Kształt macierzy kowariancji (wynik): {covariance_matrix.shape}")

=== PODSUMOWANIE WYMIARÓW ===
Kształt tensora wejściowego X:        torch.Size([4, 64, 1000])
Kształt tensora średniej:             torch.Size([4, 64, 1])
Kształt macierzy kowariancji (wynik): torch.Size([4, 64, 64])


Zadanie 4 - Maskowanie wypełnienia (padding mask) w modelach językowych

In [21]:
from torch._prims_common import ShapeType
import torch.nn.functional as F

# Dane z zadania
S = torch.tensor([
    [12, 45, 89, 0, 0],
    [30, 11, 0, 0, 0],
    [5, 67, 92, 14, 1]
])

# B = 3 zadania, T = max 5 słów
B, T = S.shape
A = torch.randn(B, T, T)

# Utworzenie maski boolowskiej, S == 0
mask = (S == 0)

# Dopasowanie wymiarów, kształt zmienia się z [B, T] na [B, 1, T]
mask_expanded = mask.unsqueeze(1)

# Zmiana wartości logitów
A_masked = A.masked_fill(mask_expanded, -1e9)

# Nałożenie funkcji softmax, obliczanie wagi atencji wzdłuż ostatniego wymiaru
attention_weights = F.softmax(A_masked, dim=-1)

print("=== PODSUMOWANIE ZADANIA 4 ===")
print(f"Kształt maski początkowej:     {mask.shape}")
print(f"Kształt maski rozszerzonej:    {mask_expanded.shape}")
print(f"Kształt tensorów atencji:      {attention_weights.shape}")

print("\nWagi atencji dla pierwszego zdania (indeks 0):")

print(torch.round(attention_weights[0], decimals=4))

=== PODSUMOWANIE ZADANIA 4 ===
Kształt maski początkowej:     torch.Size([3, 5])
Kształt maski rozszerzonej:    torch.Size([3, 1, 5])
Kształt tensorów atencji:      torch.Size([3, 5, 5])

Wagi atencji dla pierwszego zdania (indeks 0):
tensor([[0.7685, 0.1838, 0.0476, 0.0000, 0.0000],
        [0.3444, 0.4975, 0.1582, 0.0000, 0.0000],
        [0.6315, 0.0766, 0.2919, 0.0000, 0.0000],
        [0.2148, 0.4667, 0.3185, 0.0000, 0.0000],
        [0.1420, 0.5179, 0.3400, 0.0000, 0.0000]])
